In [6]:
import sys

sys.path.append('../../')
import src.forecasting.models      as fm
import src.forecasting.pipelines   as fp
import src.forecasting.simulations as fsim
from itertools import product
import src.fda.kde.estimators           as kde
import src.fda.transformations.lqdt     as lqdt

In [ ]:

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# Assumes these are available in your project
# from your_ast_module import StandardizedSkewStudentT
# from your_forecasting_module import run_multivariate_forecaster


class ASTParameterEstimator:
    """
    Daily MLE estimator for the standardized Fernández-Steel skew-t model.

    State vector:
        theta = (mu, log_sigma, log_a)

    where
        sigma = exp(log_sigma)
        a     = exp(log_a)
    """

    def __init__(self, nu=8):
        self.nu = nu

    def _negative_loglikelihood(self, theta, z):

        mu = theta[0]
        sigma = np.exp(theta[1])
        a = np.exp(theta[2])

        dist = fsim.StandardizedSkewStudentT(
            a=a,
            nu=self.nu
        )

        x = (z - mu) / sigma

        loglik = np.sum(
            -np.log(sigma)
            + dist.density(x, log=True)
        )

        return -loglik

    def fit_day(self, z, x0=None):

        z = np.asarray(z)

        if x0 is None:
            x0 = np.array([
                np.mean(z),
                np.log(np.std(z)),
                0.0
            ])

        res = minimize(
            self._negative_loglikelihood,
            x0=x0,
            args=(z,),
            method="L-BFGS-B"
        )

        return res.x

    def fit(self, samples: pd.DataFrame):

        params = []

        x0 = None

        for col in samples.columns:

            z = samples[col].dropna().values

            theta = self.fit_day(
                z=z,
                x0=x0
            )

            params.append(theta)

            x0 = theta.copy()

        params = pd.DataFrame(
            params,
            columns=[
                "mu",
                "log_sigma",
                "log_a"
            ],
            index=samples.columns
        )

        return params


class ParametricDensityForecaster:
    """
    Parametric benchmark based on daily MLE estimation of an
    asymmetric Student-t distribution followed by VAR forecasting.
    """

    def __init__(
        self,
        nu=3,
        maxlags=10,
        criteria="bic"
    ):

        self.nu = nu
        self.maxlags = maxlags
        self.criteria = criteria

        self.model_estimator = None
        self.parameter_history = None

    def fit(
        self,
        samples: pd.DataFrame
    ):

        self.model_estimator = ASTParameterEstimator(
            nu=self.nu
        )

        self.parameter_history = self.model_estimator.fit(
            samples
        )

        return self

    def predict(
        self,
        horizon,
        support,
        var_lags=None
    ):

        theta_fc = fp.run_multivariate_forecaster(

            scores=self.parameter_history.values,

            maxlags_=self.maxlags,

            criteria_=self.criteria,

            h_=horizon,

            selected_nlags=var_lags

        )

        mu_fc = theta_fc[0]
        sigma_fc = np.exp(theta_fc[1])
        a_fc = np.exp(theta_fc[2])

        density_list = []
        support_list = []

        for h in range(horizon):

            dist = fm.StandardizedSkewStudentT(
                a=a_fc[h],
                nu=self.nu
            )

            x = (support - mu_fc[h]) / sigma_fc[h]

            density = dist.density(x) / sigma_fc[h]

            density_list.append(density)
            support_list.append(support)

        future_densities = pd.DataFrame(
            np.asarray(density_list).T
        )

        future_supports = pd.DataFrame(
            np.asarray(support_list).T
        )

        future_parameters = pd.DataFrame({
            "mu": mu_fc,
            "sigma": sigma_fc,
            "a": a_fc
        })

        return {

            "future_parameters": future_parameters,

            "future_supports": future_supports,

            "future_densities": future_densities

        }

In [7]:
# Define scenarios
gas_params = {
    "scenario_4": {
        "Description": "Mixture",
        "alpha": np.diag([0.04, 0.06, 0.04]),
        "beta": np.diag([0.92, 0.95, 0.94]),
    }
}

distribution_params = {
    "nu": [3]
}

keys = ["scenario", "nu"]
values = [list(gas_params.keys()), distribution_params["nu"]]

param_grid = []

for scenario, nu in product(*values):
    gas_cfg = gas_params[scenario]

    param_grid.append({
        "scenario": "___nu=".join([scenario, str(nu)]),
        "nu": nu,
        "alpha": gas_cfg["alpha"],
        "beta": gas_cfg["beta"]    
})
    
# grid for densities
x = np.linspace(-40, 40, 5001)
# number of curves (densities)
T = 301
# number of simulations (f_{N_REP,1},...,f_{N_REP,T})
N_REPS = 10
# number of samples from each f_t density
N_SAMPLES = 288

total = len(param_grid) * N_REPS

sim_database = {}
total = len(param_grid) * N_REPS

for params in param_grid:
    scenario = params["scenario"]
    # Initialize scenario level
    sim_database[scenario] = {
        "params": params,
        "replications": {}
    }
    
    for n_rep in range(N_REPS):
        # 1. Setup Model and Simulate
        gm = fsim.GasModel(alpha=params["alpha"], beta=params["beta"], nu=params["nu"])
        sim_results = gm.simulate(T=T, burn_in=300)
        
        # 2. Get Theoretical Densities
        sim_density = gm.conditional_densities(grid=x, theta_path=sim_results["theta"])
        dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=T, freq="D")
        sim_density.columns = dates
        
        # 3. Generate Samples Efficiently
        # Collect arrays first, then create DataFrame once
        samples_list = []
        for i in range(len(sim_results["theta"])):
            # Draw n samples for the theta at time i
            sample = gm.rvs(n=N_SAMPLES, theta=sim_results["theta"][i])
            samples_list.append(sample)
        
        # Create DataFrame: each column is a time step, each row a sample
        df_samples = pd.DataFrame(np.array(samples_list).T, columns=dates)
        
        # 4. Store in Database
        sim_database[scenario]["replications"][n_rep] = {
            "theta": sim_results["theta"],
            "densities": sim_density,
            "samples": df_samples
        }

returns_df = sim_database['scenario_4___nu=3']['replications'][0]['samples']
# bandwidths
kde_bw_params = {"method": "rot", "kernel": "gaussian", "sigma_robust":False}

df_h = kde.df_bandwidth_selector(returns_df, **kde_bw_params)    
kde_params = {k: v for k, v in kde_bw_params.items() if k in ['kernel', 'df']}

# kdes
df_grids, df_densities = kde.df_to_kde(
    X=returns_df, 
    h=df_h, 
    normalize_densities=False,
    **kde_params
)

In [9]:
returns_df

,2025-08-24,2025-08-25,2025-08-26,2025-08-27,2025-08-28,2025-08-29,2025-08-30,2025-08-31,2025-09-01,2025-09-02,...,2026-06-11,2026-06-12,2026-06-13,2026-06-14,2026-06-15,2026-06-16,2026-06-17,2026-06-18,2026-06-19,2026-06-20
0,-2.019649,-1.053287,1.053597,1.440869,0.115419,-0.129600,0.021939,-0.513328,0.563328,0.265309,...,-1.477695,-1.084980,-0.456147,0.231344,-0.973394,-0.088470,-0.864264,-0.228993,-0.930642,0.015282
1,0.657304,0.616018,0.251999,0.185692,-0.017259,0.680620,0.346968,0.446393,-0.145285,-0.399595,...,0.814522,0.301549,-1.248459,-0.785296,-0.403858,-0.639389,-0.863671,0.282703,0.167501,0.142926
2,1.983934,-0.057153,-0.060547,-0.107197,0.525435,0.497552,0.893050,0.400426,0.399458,0.119402,...,1.762336,1.144393,-0.013025,-0.342794,-0.102879,-0.273056,6.256526,-0.950808,-1.702922,0.605089
3,0.062923,-0.209896,1.876260,0.019893,0.152547,-0.176825,0.325724,0.817345,0.997742,0.300431,...,-0.770536,-0.271812,0.913625,0.454253,0.301076,-0.058995,-1.162127,0.504619,0.510839,0.414261
4,-0.743389,-1.503012,0.157500,-1.614510,0.562113,-0.060343,0.650694,0.453466,0.043852,0.876599,...,0.537543,-1.025390,-1.068952,-0.260953,-0.418269,0.000740,1.685548,-0.271468,0.618702,1.022113
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,-1.921421,-0.502630,-0.041360,0.284656,-0.033747,0.205543,0.219034,0.320713,0.121620,0.909119,...,-0.252467,-0.109305,-0.333883,-0.954921,0.375771,-0.742565,-0.103960,3.454208,-0.824274,-0.900902
284,0.557195,0.690629,0.970841,0.043106,-0.857095,0.235611,0.700632,0.230631,0.205231,0.309401,...,0.655787,-0.636851,-2.237347,0.342357,-0.809771,-0.158533,1.612485,0.858556,0.066561,-0.527706
285,0.757675,0.798692,4.064957,0.225156,-0.525642,-1.070319,0.363766,-0.191687,0.083580,0.114454,...,-0.770366,-0.007836,-0.382239,5.740307,-0.779943,-0.294172,-3.820647,-0.812305,0.512909,-0.320291
286,0.449123,0.788368,0.441763,0.346410,-1.629376,-0.141438,0.641948,0.693957,-0.630387,0.021742,...,-0.454937,-0.102861,-0.419747,0.850321,0.285694,0.333301,-0.760549,-1.001635,-0.677395,-0.540161
